# **Data preprocessing**

In [ ]:
!pip install spacy
!python -m spacy download en_core_web_sm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 84.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import spacy
import json
import ast

def generate_dependency_tree(sentence, nlp):
    """
    Generates a dependency tree string for a given sentence using spaCy.

    Args:
        sentence (str): The input sentence.
        nlp: The loaded spaCy language model.

    Returns:
        str: A string representing the dependency tree.
    """
    doc = nlp(sentence)
    tree_lines = []
    for token in doc:
        # Format: token-head->dependency_label
        line = f"{token.text}-{token.head.text}->{token.dep_}"
        tree_lines.append(line)
    return "\n".join(tree_lines)

def create_jsonl_file(input_txt_path, output_json_path):
    """
    Reads the raw text data, processes it, generates dependency trees,
    and writes the final structured data to a JSONL file.

    Args:
        input_txt_path (str): Path to the input .txt file (e.g., 'train.txt').
        output_json_path (str): Path for the output .json file (e.g., 'rest15-dep.json').
    """
    try:
        # Load the small English model for spaCy.
        # You might need to download it first by running:
        # python -m spacy download en_core_web_sm
        nlp = spacy.load("en_core_web_sm")
        print(f"Successfully loaded spaCy model 'en_core_web_sm'.")
    except OSError:
        print("Spacy model 'en_core_web_sm' not found.")
        print("Please run: python -m spacy download en_core_web_sm")
        return

    # The constant system message for the chat format
    system_message = {
        "role": "system",
        "content": "Task: Extract aspect-sentiment quadruples from the query: [aspect, category, sentiment, opinion_word].\n\n"
    }

    processed_records = []

    print(f"Starting preprocessing for '{input_txt_path}'...")

    with open(input_txt_path, 'r', encoding='utf-8') as f_in:
        for line in f_in:
            line = line.strip()
            if '####' not in line:
                continue

            # 1. Split sentence from annotations
            parts = line.split('####')
            sentence = parts[0]
            annotations_str = parts[1]

            # 2. Safely parse the annotation string into a Python list
            try:
                annotations = ast.literal_eval(annotations_str)
            except (ValueError, SyntaxError) as e:
                print(f"Warning: Could not parse annotation string: {annotations_str}. Error: {e}")
                continue

            # 3. Generate the dependency tree for the sentence
            dependency_tree = generate_dependency_tree(sentence, nlp)

            # 4. Construct the user message content
            user_content = f"\n\nQuery:\n{sentence}\nDepency-tree of query:\n{dependency_tree}\nLABELS:\n"

            # 5. Assemble the final JSON object structure
            record = {
                "messages": [
                    system_message,
                    {"role": "user", "content": user_content},
                    # The assistant's content is the string representation of the annotations
                    {"role": "assistant", "content": json.dumps(annotations)}
                ]
            }
            processed_records.append(record)

    # 6. Write all processed records to the output JSONL file
    with open(output_json_path, 'w', encoding='utf-8') as f_out:
        for record in processed_records:
            f_out.write(json.dumps(record) + '\n')

    print(f"Preprocessing complete. Output saved to '{output_json_path}'.")
    print(f"Processed {len(processed_records)} records.")

if __name__ == '__main__':
    # Define file paths
    train_input = 'train.txt'
    train_output = 'rest15-dep-train.json'
    test_input = 'test.txt'
    test_output = 'rest15-dep-test.json'

    # Process both the training and test datasets
    create_jsonl_file(train_input, train_output)
    print("-" * 20)
    create_jsonl_file(test_input, test_output)


Successfully loaded spaCy model 'en_core_web_sm'.
Starting preprocessing for 'train.txt'...
Preprocessing complete. Output saved to 'rest15-dep-train.json'.
Processed 834 records.
--------------------
Successfully loaded spaCy model 'en_core_web_sm'.
Starting preprocessing for 'test.txt'...
Preprocessing complete. Output saved to 'rest15-dep-test.json'.
Processed 537 records.


# **Fine tuning Process**

In [ ]:
!pip install -qU transformers trl peft bitsandbytes accelerate datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.7/564.7 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 5.9 MB/s eta 0:00:00


In [ ]:
import os
from typing import Dict, List, Any
import json
import random
from dataclasses import dataclass

import torch
import bitsandbytes as bnb
from datasets import Dataset, DatasetDict, load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)

from trl import SFTTrainer, SFTConfig

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Torch: 2.8.0+cu126
CUDA available: False


In [ ]:
def load_jsonl(path: str) -> List[Dict[str, Any]]:
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

train_rows = load_jsonl("rest15-dep-train.json")
val_rows   = load_jsonl("rest15-dep-test.json")

raw_ds = DatasetDict({
    "train": Dataset.from_list(train_rows),
    "validation": Dataset.from_list(val_rows),
})

raw_ds

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 834
    })
    validation: Dataset({
        features: ['messages'],
        num_rows: 537
    })
})

In [ ]:
ASSISTANT_PREFIX = "### Assistant:\n"
USER_PREFIX      = "### User:\n"
SYSTEM_PREFIX    = "### System:\n"

def messages_to_prompt(example: Dict[str, Any]) -> Dict[str, str]:
    msgs = example["messages"]

    sys = next((m["content"] for m in msgs if m["role"] == "system"), "")
    usr = next((m["content"] for m in msgs if m["role"] == "user"), "")
    ans = next((m["content"] for m in msgs if m["role"] == "assistant"), "")

    prompt = (
        f"{SYSTEM_PREFIX}{sys.strip()}\n\n"
        f"{USER_PREFIX}{usr.strip()}\n\n"
        f"{ASSISTANT_PREFIX}"
    )
    # The "completion" is the assistant text (labels)
    completion = ans.strip()

    # We store the merged text for SFTTrainer (prompt+completion)
    # The collator will mask out everything up to ASSISTANT_PREFIX.
    return {
        "text": prompt + completion,
    }

proc_ds = raw_ds.map(messages_to_prompt, remove_columns=raw_ds["train"].column_names)
proc_ds

Map:   0%|          | 0/834 [00:00<?, ? examples/s]

Map:   0%|          | 0/537 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 834
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 537
    })
})

In [ ]:
HF_TOKEN = "YOUR_HUGGINGFACE_TOKEN_HERE"
token-name = 'YOUR_HUGGINGFACE_TOKEN-name_HERE'

In [ ]:
from huggingface_hub import login
from google.colab import userdata

try:
    # Get the token from Colab secrets
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        login(token=hf_token)
        print("Hugging Face login successful!")
    else:
        print("HF_TOKEN not found in Colab secrets.")
except Exception as e:
    print(f"An error occurred during Hugging Face login: {e}")

Hugging Face login successful!


In [ ]:
model_name = "meta-llama/Meta-Llama-3-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True,

)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,

)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,

)

# Training stability tweaks
model.config.use_cache = False  # important for gradient checkpointing
print("model dtype:", next(model.parameters()).dtype)
print("model device:", next(model.parameters()).device)

tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

In [ ]:
model = prepare_model_for_kbit_training(model)
print("Applied prepare_model_for_kbit_training.")

In [ ]:
import torch.nn as nn

def find_linear_module_names(model) -> List[str]:
    linear_like = (nn.Linear, bnb.nn.Linear4bit, bnb.nn.Linear8bitLt)
    names = set()
    for name, module in model.named_modules():
        if isinstance(module, linear_like):
            leaf_name = name.split(".")[-1]
            if leaf_name != "lm_head":
                names.add(leaf_name)
    # Common practice is to return unique leaf names (module class fields)
    return sorted(list(names))

target_modules = find_linear_module_names(model)
print("LoRA target modules:", target_modules[:20], " ... total:", len(target_modules))

lora_config = LoraConfig(
    r=256,
    lora_alpha=512,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=target_modules,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Robust CompletionOnlyDataCollator that accepts:
# - features with "text" (string)
# - features already tokenized ("input_ids", optional "attention_mask")
# - features that are plain strings
import torch
from typing import List, Dict, Optional, Any

class CompletionOnlyDataCollator:
    def __init__(
        self,
        tokenizer,
        response_template: str,
        padding: bool = True,
        truncation: bool = True,
        max_length: Optional[int] = None,
        return_tensors: str = "pt",
        ignore_index: int = -100,
    ):
        self.tokenizer = tokenizer
        self.response_template = response_template
        self.response_ids = tokenizer.encode(response_template, add_special_tokens=False)
        self.padding = padding
        self.truncation = truncation
        self.max_length = max_length
        self.return_tensors = return_tensors
        self.ignore_index = ignore_index
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        self.pad_id = tokenizer.pad_token_id

    @staticmethod
    def _find_subsequence(hay: List[int], needle: List[int]) -> Optional[int]:
        if len(needle) == 0:
            return None
        for i in range(len(hay) - len(needle) + 1):
            if hay[i : i + len(needle)] == needle:
                return i
        return None

    @staticmethod
    def _to_list(x: Any) -> List[int]:
        # Convert tensors/np arrays/ints to python list of ints
        if isinstance(x, torch.Tensor):
            return x.tolist()
        if isinstance(x, (list, tuple)):
            return list(x)
        try:
            return list(x)  # fallback; may raise
        except Exception:
            return [int(x)]

    def __call__(self, features: List[Dict]) -> Dict[str, torch.Tensor]:
        if len(features) == 0:
            raise ValueError("No features provided to collator")

        f0 = features[0]
        # Case A: features contain 'text' strings
        if isinstance(f0, dict) and "text" in f0:
            texts = [f["text"] for f in features]
            tokenized = self.tokenizer(
                texts,
                padding=self.padding,
                truncation=self.truncation,
                max_length=self.max_length,
                return_attention_mask=True,
            )
            input_ids_list = tokenized["input_ids"]
            attention_mask_list = tokenized["attention_mask"]

        # Case B: features already tokenized (input_ids present)
        elif isinstance(f0, dict) and "input_ids" in f0:
            input_ids_list = []
            attention_mask_list = []
            for f in features:
                ids = f["input_ids"]
                ids = self._to_list(ids)
                # ensure flattened 1D list (sometimes huggingface stores as list of ints)
                if isinstance(ids[0], list):
                    # choose first row if 2D (unexpected) — flatten
                    ids = [int(x) for sub in ids for x in sub]
                input_ids_list.append([int(x) for x in ids])

                if "attention_mask" in f:
                    am = self._to_list(f["attention_mask"])
                    attention_mask_list.append([int(x) for x in am])
                else:
                    attention_mask_list.append([1] * len(ids))

        # Case C: features are raw strings (trainer might pass strings)
        elif isinstance(f0, str):
            texts = list(features)
            tokenized = self.tokenizer(
                texts,
                padding=self.padding,
                truncation=self.truncation,
                max_length=self.max_length,
                return_attention_mask=True,
            )
            input_ids_list = tokenized["input_ids"]
            attention_mask_list = tokenized["attention_mask"]

        else:
            raise ValueError(
                "Unrecognized feature format for collator. Feature keys: "
                f"{list(f0.keys()) if isinstance(f0, dict) else type(f0)}"
            )

        # Make labels copy and mask pre-response tokens
        labels_list = [list(ids) for ids in input_ids_list]
        not_found_count = 0
        for i, ids in enumerate(input_ids_list):
            start = self._find_subsequence(ids, self.response_ids)
            if start is None:
                not_found_count += 1
                # fallback: find last occurrence of the response-template-first-token if possible
                if len(self.response_ids) > 0 and self.response_ids[0] in ids:
                    # try to find last index of first token and use that as approximate start
                    idxs = [j for j, tok in enumerate(ids) if tok == self.response_ids[0]]
                    start = idxs[-1]
                else:
                    # if we cannot find any marker, mask nothing (safer than masking whole example)
                    start = 0
                    # we will warn once after loop
            # Set labels before start to ignore_index
            for j in range(0, start):
                labels_list[i][j] = self.ignore_index

        if not_found_count > 0:
            # print a short helpful message (only when some were not found)
            print(
                f"Warning: response_template not found in {not_found_count}/{len(input_ids_list)} examples. "
                "These examples had no masking before the assistant reply (you may want to inspect them)."
            )

        # Pad to max length in batch
        max_len = max(len(ids) for ids in input_ids_list)
        for i in range(len(input_ids_list)):
            pad_len = max_len - len(input_ids_list[i])
            if pad_len > 0:
                input_ids_list[i] = input_ids_list[i] + [self.pad_id] * pad_len
                attention_mask_list[i] = attention_mask_list[i] + [0] * pad_len
                labels_list[i] = labels_list[i] + [self.ignore_index] * pad_len

        batch = {
            "input_ids": torch.tensor(input_ids_list, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask_list, dtype=torch.long),
            "labels": torch.tensor(labels_list, dtype=torch.long),
        }
        return batch

In [ ]:
collator = CompletionOnlyDataCollator(tokenizer=tokenizer, response_template=ASSISTANT_PREFIX)

In [ ]:
print(proc_ds["train"][0].keys())

In [ ]:
output_dir = "llama3-asqp-lora"

sft_config = SFTConfig(
    output_dir=output_dir,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    optim="paged_adamw_32bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    bf16=True,
    logging_steps=10,
    save_steps=100,
    eval_steps=10,
    num_train_epochs=4,       # increase for real training
    report_to="none",         # set to "wandb" or "tensorboard" if you use them
)

trainer = SFTTrainer(
    model=model,
    train_dataset=proc_ds["train"],
    eval_dataset=proc_ds["validation"],
    data_collator=collator,
    args=sft_config,
)


In [ ]:
changed = 0
for n, p in model.named_parameters():
    if ("lora" in n.lower() or "adapter" in n.lower() or "alpha" in n.lower() or "peft" in n.lower()):
        if not p.requires_grad:
            p.requires_grad = True
            changed += 1
print("Forced requires_grad=True on", changed, "params")
print("Trainable params now:", sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
# Force trainer to use the already-prepared peft model
# (Only do this if the "model" var above shows non-zero trainable params)
trainer.model = model
# if trainer uses accelerator/ device mapping, move the model to the right device:
try:
    device = next(trainer.model.parameters()).device
except Exception:
    import torch
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
trainer.model = trainer.model.to(device)
print("Replaced trainer.model with your `model` and moved to device:", device)

# Re-print summary
from peft import PeftModel
def quick_print():
    trainable = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in trainer.model.parameters())
    print("After replacement: trainer.model trainable params:", trainable, "/", total)
quick_print()

In [ ]:
# ---------------------------
# metrics & evaluation helper
# ---------------------------
import re
import json
from typing import List, Tuple, Set, Dict
import math
import torch
from tqdm.auto import tqdm

# 1) Robust JSON-extraction from model-generated text
_JSON_ARRAY_RE = re.compile(r"(\[.*\])", re.DOTALL)

def extract_json_from_text(text: str) -> List:
    """
    Finds the first JSON array in text and returns the parsed Python object.
    If parsing fails or not found, returns [].
    """
    if text is None:
        return []
    m = _JSON_ARRAY_RE.search(text)
    if not m:
        # no JSON-like array found: maybe model returned plain brackets spaced or single quotes
        # try replacing single quotes to double quotes and re-search
        text2 = text.replace("'", '"')
        m2 = _JSON_ARRAY_RE.search(text2)
        if not m2:
            return []
        try:
            return json.loads(m2.group(1))
        except Exception:
            return []
    try:
        return json.loads(m.group(1))
    except Exception:
        # attempt to fix single quotes -> double quotes and retry
        try:
            fixed = m.group(1).replace("'", '"')
            return json.loads(fixed)
        except Exception:
            return []

# 2) Normalization helpers for quadruples
def normalize_quad(q):
    # expect q like ['Food','food quality','positive','great']
    if not isinstance(q, (list, tuple)) or len(q) != 4:
        return None
    return tuple([str(x).strip().lower() for x in q])

def quads_list_to_set(quads) -> Set[Tuple[str,str,str,str]]:
    out = set()
    if not isinstance(quads, (list,tuple)):
        return out
    for q in quads:
        nq = normalize_quad(q)
        if nq is not None:
            out.add(nq)
    return out

# 3) micro F1 across dataset
def preds_refs_to_micro_f1(preds: List[List], refs: List[List]) -> Dict[str, float]:
    """
    preds, refs: list of parsed JSON arrays per example (each is a list of quads)
    returns dict: precision, recall, f1
    counting each quadruple as an item (micro-averaged).
    """
    tp = 0
    fp = 0
    fn = 0
    for p_list, r_list in zip(preds, refs):
        p_set = quads_list_to_set(p_list)
        r_set = quads_list_to_set(r_list)
        tp += len(p_set & r_set)
        fp += len(p_set - r_set)
        fn += len(r_set - p_set)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1  = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    return {"precision": prec, "recall": rec, "f1": f1, "tp": tp, "fp": fp, "fn": fn}

# 4) function to perform batched generation over eval_dataset and compute metrics
def eval_generate_and_score(
    trainer,
    tokenizer,
    eval_dataset,
    prompt_field="text",
    assistant_prefix=ASSISTANT_PREFIX,
    batch_size=8,
    max_new_tokens=256,
    gen_kwargs=None,
    device=None,
):
    """
    Generates completions for every example in eval_dataset and computes quad-level micro F1.
    - trainer: the SFTTrainer instance (used to access model.device and model.generate)
    - tokenizer: the tokenizer
    - eval_dataset: HF dataset or list of dicts with `prompt_field` containing prompt+assistant_marker
    - assistant_prefix: the exact marker used in training to mark assistant start (must match)
    - Returns a dict of computed metrics and optionally raw preds/refs
    """
    if device is None:
        device = trainer.model.device

    # build prompts: keep only things up to ASSISTANT_PREFIX (include the marker)
    prompts = []
    refs = []
    for ex in eval_dataset:
        # ex may be dict with 'text' containing prompt+completion OR separate fields
        if isinstance(ex, dict) and prompt_field in ex:
            full = ex[prompt_field]
        else:
            # fallback: if dataset has 'messages' field like original, reconstruct prompt
            if isinstance(ex, dict) and "messages" in ex:
                msgs = ex["messages"]
                sys = next((m["content"] for m in msgs if m["role"] == "system"), "")
                usr = next((m["content"] for m in msgs if m["role"] == "user"), "")
                ans = next((m["content"] for m in msgs if m["role"] == "assistant"), "")
                full = f"{SYSTEM_PREFIX}{sys.strip()}\n\n{USER_PREFIX}{usr.strip()}\n\n{ASSISTANT_PREFIX}{ans.strip()}"
            else:
                # last resort: stringify ex
                full = str(ex)
        # split widely on ASSISTANT_PREFIX to separate prompt and reference
        if assistant_prefix in full:
            prompt_part, completion_part = full.split(assistant_prefix, 1)
            prompt = prompt_part + assistant_prefix
            ref_text = completion_part.strip()
        else:
            # if marker missing, treat entire text as prompt and ref empty
            prompt = full + assistant_prefix
            ref_text = ""
        prompts.append(prompt)
        # try parse ref_text as JSON array (if not parseable -> assume [])
        refs.append(extract_json_from_text(assistant_prefix + ref_text) if ref_text else [])
    # batched generation
    all_preds = []
    model = trainer.model
    model.eval()
    gen_config = dict(
        max_new_tokens=max_new_tokens,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )
    if gen_kwargs:
        gen_config.update(gen_kwargs)

    # iterate by batch
    for i in tqdm(range(0, len(prompts), batch_size), desc="Eval generate"):
        batch_prompts = prompts[i : i + batch_size]
        tokenized = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
        ).to(device)

        with torch.no_grad():
            generated = model.generate(
                **tokenized,
                **gen_config
            )
        # decode and keep only assistant completion portion
        for out in generated:
            text = tokenizer.decode(out, skip_special_tokens=True)
            # if ASSISTANT_PREFIX in produced text, take content after it
            if assistant_prefix in text:
                completion = text.split(assistant_prefix, 1)[-1].strip()
            else:
                # fallback: remove prompt prefix if present
                completion = text
            parsed = extract_json_from_text(completion)
            all_preds.append(parsed)

    # compute micro F1
    metrics = preds_refs_to_micro_f1(all_preds, refs)
    return {"metrics": metrics, "preds": all_preds, "refs": refs}

# ---------------------------
# Integration: attach to trainer
# ---------------------------

# 1) Try passing compute_metrics to SFTTrainer if supported:
def compute_metrics_wrapper(eval_preds=None):
    # This wrapper is not used directly by our generation procedure but left for API compatibility
    # We'll instead use eval_generate_and_score below in the callback approach.
    return {}

# 2) Callback fallback: run this on evaluation and log metrics
from transformers import TrainerCallback, TrainingArguments, TrainerState, TrainerControl

class F1EvalCallback(TrainerCallback):
    """
    On evaluation (called by trainer at evaluation steps), run generation over the eval dataset
    and compute F1, then update trainer logs.
    """
    def __init__(self, eval_dataset, tokenizer, batch_size=8, max_new_tokens=256, prompt_field="text"):
        self.eval_dataset = eval_dataset
        self.tokenizer = tokenizer
        self.batch_size = batch_size
        self.max_new_tokens = max_new_tokens
        self.prompt_field = prompt_field

    def on_evaluate(self, args: TrainingArguments, state: TrainerState, control: TrainerControl, **kwargs):
        # kwargs may include 'model' or 'trainer'; try to use trainer if available
        trainer = kwargs.get("trainer", None)
        if trainer is None:
            # sometimes model passed but not trainer. try to fetch global 'trainer' var if present
            try:
                trainer = globals().get("trainer", None)
            except Exception:
                trainer = None
        if trainer is None:
            print("F1EvalCallback: trainer not available in on_evaluate kwargs; skipping F1 computation.")
            return

        results = eval_generate_and_score(
            trainer=trainer,
            tokenizer=self.tokenizer,
            eval_dataset=self.eval_dataset,
            prompt_field=self.prompt_field,
            batch_size=self.batch_size,
            max_new_tokens=self.max_new_tokens,
        )
        metrics = results["metrics"]
        # Add to the logs so they appear alongside eval loss etc.
        to_log = {
            "eval_f1": metrics["f1"],
            "eval_precision": metrics["precision"],
            "eval_recall": metrics["recall"],
            "eval_tp": metrics["tp"],
            "eval_fp": metrics["fp"],
            "eval_fn": metrics["fn"],
        }
        # The trainer may expose a `log` method
        try:
            trainer.log(to_log)
        except Exception:
            # fallback to printing
            print("Evaluation metrics:", to_log)

# ---------------------------
# How to use
# ---------------------------

# Option A (preferred if SFTTrainer accepts compute_metrics): pass compute_metrics when constructing trainer.
# E.g.:
# trainer = SFTTrainer(..., compute_metrics=compute_metrics_wrapper, ...)

# Option B (fallback): Add the callback and ensure evaluation is scheduled (eval_steps or evaluation_strategy set).
# Example:
f1_callback = F1EvalCallback(eval_dataset=proc_ds["validation"], tokenizer=tokenizer, batch_size=4, max_new_tokens=128)
# if SFTTrainer supports add_callback:
try:
    trainer.add_callback(f1_callback)
except Exception:
    # if not supported, pass callbacks at trainer creation or use Trainer API-compatible hook
    print("trainer.add_callback not supported; try passing callbacks in SFTTrainer constructor or run manual eval after training.")

# If add_callback worked, the callback will compute and log eval_f1 at each evaluation.
# If it didn't, you can always call the eval manually after training:
# results = eval_generate_and_score(trainer, tokenizer, proc_ds["validation"], batch_size=4)
# print(results["metrics"])


In [ ]:
trainer.train()

In [ ]:
# ---- Fixes for: padding warning + generation flags warning + dtype mismatch ----
import torch, re, json
from tqdm.auto import tqdm

# CONFIG (edit as needed)
MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"
ASSISTANT_PREFIX = "### Assistant:\n"
SYSTEM_PREFIX = "### System:\n"
USER_PREFIX = "### User:\n"

GEN_BATCH_SIZE = 8
MAX_NEW_TOKENS = 128
DO_SAMPLE = False           # set True if you want sampling
TEMPERATURE = 0.1
TOP_P = 0.9

# 1) tokenizer fixes: left padding and pad token
tokenizer.padding_side = "left"            # IMPORTANT for decoder-only models
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.pad_token_id

print("tokenizer.padding_side:", tokenizer.padding_side, "pad_token_id:", tokenizer.pad_token_id)

# 2) ensure eval_model exists (from trainer.model or model var)
eval_model = getattr(trainer, "model", None) or globals().get("model")
if eval_model is None:
    raise RuntimeError("No model found in variables `trainer` or `model`.")

device = next(eval_model.parameters()).device
print("Model device before adjustments:", device, "model dtype sample:", next(eval_model.parameters()).dtype)

# 3) Try to put model in float16 for generation (safe & common)
#    If model is a bnb 4-bit model this .to may fail or be noop; handle exceptions.
try:
    # prefer float16 for most GPUs (unless you explicitly need bfloat16)
    eval_model = eval_model.to(dtype=torch.float16)
    print("Converted model to float16 using .to(dtype=torch.float16).")
except Exception as e:
    # If conversion not possible (bnb layers), we'll try reloading with BitsAndBytesConfig below
    print("Could not cast model with .to(float16):", e)
    eval_model = getattr(trainer, "model", None) or globals().get("model")

# quick device ensure
try:
    eval_model = eval_model.to(device)
except Exception:
    pass

# check dtype again
try:
    sample_dtype = next(eval_model.parameters()).dtype
except Exception:
    sample_dtype = None
print("Model sample dtype after attempt:", sample_dtype)

# 4) If dtype mismatch persists (e.g. model params are bfloat16 but generation errors),
#    reload model with bitsandbytes using float16 compute dtype.
need_reload = False
if sample_dtype is not None and sample_dtype == torch.bfloat16:
    # many consumer GPUs don't handle bfloat16 compute in bnb; prefer float16 unless you are on H100
    print("Model is bfloat16. If you get dtype errors on your GPU, consider reloading in float16.")
    # Optionally set condition to reload always if you experienced the error:
    # need_reload = True

# If earlier you hit "expected scalar type BFloat16 but found Float" the safe reload uses float16:
if 'expected scalar type BFloat16 but found Float' in globals().get("__last_exception__", ""):
    need_reload = True

# OPTIONAL: force reload to ensure compute dtype = float16 (uncomment if you want to always reload)
# need_reload = True

if need_reload:
    print("Reloading model with BitsAndBytesConfig compute dtype=float16 (this may take time)...")
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,  # float16 compute dtype for broad GPU support
        bnb_4bit_use_double_quant=True,
    )
    eval_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    # If you used PEFT/LoRA, re-attach or load adapter afterwards (not shown).
    print("Reload complete. New model dtype sample:", next(eval_model.parameters()).dtype)

eval_model.eval()
device = next(eval_model.parameters()).device
print("Using model on device:", device, "dtype:", next(eval_model.parameters()).dtype)

# 5) Build gen_kwargs properly (avoid passing sampling args when not sampling)
gen_kwargs = dict(
    max_new_tokens=MAX_NEW_TOKENS,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
    use_cache=False, # Explicitly set use_cache to False for generation
)
if DO_SAMPLE:
    gen_kwargs.update({"do_sample": True, "temperature": TEMPERATURE, "top_p": TOP_P})
else:
    gen_kwargs.update({"do_sample": False})

print("Generation kwargs:", gen_kwargs)

# 6) Example batched generation (decoding + JSON parse) — same logic as previous eval code
_JSON_ARRAY_RE = re.compile(r"(\[.*\])", re.DOTALL)
def extract_json_array_safe(text):
    if text is None:
        return []
    m = _JSON_ARRAY_RE.search(text)
    if not m:
        t2 = text.replace("'", '"')
        m2 = _JSON_ARRAY_RE.search(t2)
        if not m2:
            return []
        try:
            return json.loads(m2.group(1))
        except Exception:
            return []
    try:
        return json.loads(m.group(1))
    except Exception:
        try:
            return json.loads(m.group(1).replace("'", '"'))
        except Exception:
            return []

# pick eval dataset split
if "test" in proc_ds:
    eval_ds = proc_ds["test"]
else:
    eval_ds = proc_ds["validation"]

prompts = []
refs = []
for ex in eval_ds:
    if isinstance(ex, dict) and "text" in ex:
        full = ex["text"]
        if ASSISTANT_PREFIX in full:
            p, c = full.split(ASSISTANT_PREFIX, 1)
            prompts.append(p + ASSISTANT_PREFIX)
            refs.append(extract_json_array_safe(c))
        else:
            prompts.append(full + ASSISTANT_PREFIX)
            refs.append([])
    elif isinstance(ex, dict) and "messages" in ex:
        msgs = ex["messages"]
        sys = next((m["content"] for m in msgs if m["role"]=="system"), "")
        usr = next((m["content"] for m in msgs if m["role"]=="user"), "")
        ans = next((m["content"] for m in msgs if m["role"]=="assistant"), "")
        prompts.append(f"{SYSTEM_PREFIX}{sys.strip()}\n\n{USER_PREFIX}{usr.strip()}\n\n{ASSISTANT_PREFIX}")
        refs.append(extract_json_array_safe(ans))
    else:
        prompts.append(str(ex) + ASSISTANT_PREFIX)
        refs.append([])

# batched generation
all_preds = []
for i in tqdm(range(0, len(prompts), GEN_BATCH_SIZE), desc="Gen"):
    batch_prompts = prompts[i:i+GEN_BATCH_SIZE]
    tokenized = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True).to(device)
    # ensure attention_mask is integer type (long) - avoid unexpected float tensors
    if tokenized.get("attention_mask") is not None and tokenized["attention_mask"].dtype != torch.long:
        tokenized["attention_mask"] = tokenized["attention_mask"].long()
    with torch.no_grad():
        out = eval_model.generate(**tokenized, **gen_kwargs)
    for seq in out:
        text = tokenizer.decode(seq, skip_special_tokens=True)
        if ASSISTANT_PREFIX in text:
            completion = text.split(ASSISTANT_PREFIX, 1)[-1].strip()
        else:
            completion = text
        parsed = extract_json_array_safe(completion)
        all_preds.append(parsed)

# sanity
print("Generated", len(all_preds), "examples; refs:", len(refs))
# compute micro-F1 as before (user already has micro_f1 function)
def quads_list_to_set(quads):
    s = set()
    if not isinstance(quads, (list,tuple)): return s
    for q in quads:
        if isinstance(q, (list,tuple)) and len(q)==4:
            s.add(tuple(str(x).strip().lower() for x in q))
    return s

tp = fp = fn = 0
for p, r in zip(all_preds, refs):
    pset = quads_list_to_set(p)
    rset = quads_list_to_set(r)
    tp += len(pset & rset)
    fp += len(pset - rset)
    fn += len(rset - pset)
prec = tp/(tp+fp) if tp+fp>0 else 0.0
rec = tp/(tp+fn) if tp+fn>0 else 0.0
f1 = 2*prec*rec/(prec+rec) if prec+rec>0 else 0.0
print(f"Precision={prec:.4f} Recall={rec:.4f} F1={f1:.4f} (tp={tp} fp={fp} fn={fn})")

# Save predictions
with open("test_predictions.jsonl", "w", encoding="utf-8") as fout:
    for ex, p, r in zip(eval_ds, all_preds, refs):
        fout.write(json.dumps({"input": ex, "pred": p, "ref": r}, ensure_ascii=False) + "\n")
print("Saved test_predictions.jsonl")